# tabular_classification-iris-mlp-pytorch

Multi-class iris flower classification using a feed-forward MLP, told through `nnx`.

# 1. Overview

## 1.1 Task & motivation

Iris flower species recognition is a 75-year-old benchmark — three species (*setosa*, *versicolor*, *virginica*), four morphometric features (sepal length / width, petal length / width), 150 labeled samples evenly distributed across the classes. It's the canonical "small clean tabular classification problem" — small enough that a CPU re-run completes in seconds, clean enough that the *modeling* choices show up clearly without dataset noise drowning them out.

## 1.2 Dataset choice

The source lesson (Virginia Tech CS5644 `MLP-updated.ipynb`) used `sklearn.MLPClassifier` with 5-fold cross-validation and MinMax scaling. We keep iris and the MinMax scaler. We replace sklearn's blackbox MLP with `nnx.NNModel` so the per-epoch loss curves, validation tracking, and confusion matrices become first-class outputs of the notebook. We trade 5-fold CV for a 70 / 15 / 15 train/val/test split — at 150 samples, both regimes give comparable variance estimates, and the held-out-test split lets the candidate-comparison verdict in §6 sit on a single confusion matrix per candidate.

## 1.3 Model family

A feed-forward neural network with `nnx.FeedFwdNN` (4 input units → optional hidden layers → 3 output units, cross-entropy loss, Adam optimizer). We sweep three candidate topologies in §4 to test the *a priori* hypothesis that iris's separability is so good a linear classifier already nearly saturates it — and to quantify exactly how much (if anything) a hidden layer adds.

## 1.4 Libraries used

| Library | Used for |
|---|---|
| `nnx` | Training loop (`NNModel.train`), tabular data plumbing (`NNTabularDataset`), visualization (`VisUtils.confusion_matrix`, `VisUtils.multi_line_plot`), reproducibility (`set_seed`) |
| `torch` | Tensor backbone |
| `pandas`, `numpy`, `scikit-learn` | Iris loader, MinMax scaling, metrics, train/val/test split |
| `seaborn`, `matplotlib` | Pairplot + per-class diagnostic plots |

## 1.5 Headline result preview

The linear baseline (`hidden_dims=[]`) gets the reader to ~95 %+ test accuracy on iris. The §6 verdict will quantify whether the one-hidden-layer (`[8]`) and two-hidden-layer (`[16, 8]`) candidates close the remaining gap or just add capacity that doesn't pay for itself.


In [ ]:
SMOKE_TEST = 0
SMOKE_TEST_EPOCHS = 5


In [ ]:
# Parameters
SMOKE_TEST = 1


# 2. Environment & Setup

## 2.1 Imports

Imports are split between (a) the data + sklearn stack (iris loader, MinMax scaler, metrics, train/val/test split) and (b) the `nnx` flat re-exports introduced in the 21-commit hop (`NNModel`, `NNParams`, `NNModelParams`, `NNTrainParams`, `NNOptimParams`, `NNTabularDataset`, `Devices`, `Losses`, `Nets`, `Optims`, `set_seed`, `VisUtils`).

## 2.2 Reproducibility

`nnx.set_seed(0)` pins Python `random`, NumPy, PyTorch CPU (and CUDA if present), and cuDNN deterministic flags. We also pin `np.random.default_rng(0)` for the train/val/test split.

## 2.3 Configuration / hyperparameters

`SMOKE_TEST` papermill parameter cell controls fast-mode epoch count (`SMOKE_TEST_EPOCHS=5` when set; full `n_epochs=300` otherwise). All three §4 candidates train under the same `NNTrainParams` so the only varying axis is `hidden_dims`.


In [ ]:
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn
import torch
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score, classification_report, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

import nnx
from nnx import (
    Devices,
    Losses,
    NNModel,
    NNModelParams,
    NNOptimParams,
    NNParams,
    NNRun,
    NNTabularDataset,
    NNTrainParams,
    Nets,
    Optims,
    VisUtils,
    set_seed,
)

print(f"python   = {sys.version.split()[0]}")
print(f"torch    = {torch.__version__}")
print(f"sklearn  = {sklearn.__version__}")
print(f"pandas   = {pd.__version__}")
print(f"numpy    = {np.__version__}")
print(f"nnx      = {nnx.__version__}")


In [ ]:
SEED = 0
set_seed(SEED)

n_epochs = SMOKE_TEST_EPOCHS if SMOKE_TEST else 300
print(f"SMOKE_TEST = {SMOKE_TEST}, n_epochs = {n_epochs}")


# 3. Data

## 3.1 Loading

Iris ships inside `sklearn.datasets.load_iris`. We bind feature columns explicitly (no shorthand `iris.feature_names` consumption) so the column names that flow into `NNTabularDataset` exactly match the column names we explore with seaborn — debugging "why is my pairplot blank?" is no fun.


In [ ]:
_iris = load_iris()
FEATURE_COLS: list[str] = [
    "sepal_length",
    "sepal_width",
    "petal_length",
    "petal_width",
]
TARGET_COL: str = "species_idx"
CLASS_NAMES: list[str] = list(_iris.target_names)

df_raw = pd.DataFrame(_iris.data, columns=FEATURE_COLS)
df_raw[TARGET_COL] = _iris.target
df_raw.head()


## 3.2 Exploration

Schema check (dtypes), summary statistics (mean/std/min/max per feature), class-balance check, and a pairplot. We're looking for three things: (a) are any features on wildly different scales (motivates MinMax), (b) is the class distribution balanced (controls whether macro-averaged metrics in §6 are meaningful), (c) are the classes linearly separable from any feature pair (anchors the *a priori* reasoning behind a `hidden_dims=[]` candidate in §4).


In [ ]:
print("--- dtypes ---")
print(df_raw.dtypes)
print()
print("--- summary statistics ---")
df_raw[FEATURE_COLS].describe().T


In [ ]:
class_counts = df_raw[TARGET_COL].value_counts().sort_index()
class_balance = pd.DataFrame(
    {
        "class_idx": class_counts.index,
        "class_name": [CLASS_NAMES[i] for i in class_counts.index],
        "count": class_counts.values,
        "fraction": (class_counts.values / len(df_raw)).round(3),
    }
)
class_balance


In [ ]:
df_plot = df_raw.copy()
df_plot["species"] = df_plot[TARGET_COL].map(dict(enumerate(CLASS_NAMES)))

g = sns.pairplot(
    df_plot,
    vars=FEATURE_COLS,
    hue="species",
    height=2.0,
    diag_kind="kde",
    palette="deep",
)
g.fig.suptitle("Iris — feature pairplot by species", y=1.02)
plt.show()


## 3.3 Cleaning & preprocessing

Three steps, in order:

1. **70 / 15 / 15 train/val/test split** with `random_state=SEED` and `stratify=` on the species label. Stratifying prevents the (rare-but-possible) case where the test split misses a whole class — at 150 samples × 15 % = ~22 test rows, a 3-class miss is a real risk with default random split.
2. **MinMax scaling** fit on the train split *only*, then applied to val + test. Petal length spans `[1.0, 6.9]` and sepal length spans `[4.3, 7.9]` — different orders of magnitude across features means the gradient steps for the wide-range feature dominate. MinMax maps every feature into `[0, 1]` so the dimensional ranges match the expected input range of `nnx`'s default LeakyReLU + linear output stack.
3. **`NNTabularDataset` construction** — three separate datasets (one per split), each wrapping the corresponding scaled DataFrame slice with `batch_sizes=(32, None, None)` (train uses 32; val + test use the full split as one batch).


In [ ]:
# 70 / 15 / 15 stratified split. train_test_split is run twice: once to
# carve off train (70%), then again on the remainder to halve it into
# val and test (15% each of the original).
X_raw = df_raw[FEATURE_COLS].to_numpy()
y_raw = df_raw[TARGET_COL].to_numpy()

X_train, X_rest, y_train, y_rest = train_test_split(
    X_raw, y_raw,
    test_size=0.30,
    random_state=SEED,
    stratify=y_raw,
)
X_val, X_test, y_val, y_test = train_test_split(
    X_rest, y_rest,
    test_size=0.50,  # half of the 30% remainder -> 15% val + 15% test
    random_state=SEED,
    stratify=y_rest,
)

# MinMax fit on train only, then applied to val + test.
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print(f"split sizes: train={len(X_train)}, val={len(X_val)}, test={len(X_test)}")
print(f"train feature ranges (post-scaling):")
print(pd.DataFrame(X_train_scaled, columns=FEATURE_COLS).describe().T[["min", "max"]])


In [ ]:
def _make_split_df(X: np.ndarray, y: np.ndarray) -> pd.DataFrame:
    """Re-assemble a scaled-feature + target DataFrame for NNTabularDataset."""
    out = pd.DataFrame(X, columns=FEATURE_COLS)
    out[TARGET_COL] = y
    return out

df_train = _make_split_df(X_train_scaled, y_train)
df_val   = _make_split_df(X_val_scaled,   y_val)
df_test  = _make_split_df(X_test_scaled,  y_test)

# NNTabularDataset accepts a single DataFrame and internally carves
# val/test slices via val_proportion/test_proportion. Since we've
# already split deterministically with sklearn's stratified split,
# pass val_proportion=0 and test_proportion=0 on the train dataset
# (so its full content becomes the train loader) and construct
# separate single-split datasets for val and test.
ds_train = NNTabularDataset(
    df=df_train,
    feature_cols=FEATURE_COLS,
    target_col=TARGET_COL,
    batch_sizes=(32, None, None),
    val_proportion=0.0,
    test_proportion=0.0,
    name_override="iris-train",
)
ds_val = NNTabularDataset(
    df=df_val,
    feature_cols=FEATURE_COLS,
    target_col=TARGET_COL,
    batch_sizes=(len(df_val), None, None),
    val_proportion=0.0,
    test_proportion=0.0,
    name_override="iris-val",
)
ds_test = NNTabularDataset(
    df=df_test,
    feature_cols=FEATURE_COLS,
    target_col=TARGET_COL,
    batch_sizes=(len(df_test), None, None),
    val_proportion=0.0,
    test_proportion=0.0,
    name_override="iris-test",
)

train_loader = ds_train.train_loader
val_loader   = ds_val.train_loader   # full-split single-batch loader
test_loader  = ds_test.train_loader  # full-split single-batch loader

print(f"ds_train: input_dim={ds_train.input_dim}, output_dim={ds_train.output_dim}, name={ds_train.name}")
print(f"ds_val:   input_dim={ds_val.input_dim},   output_dim={ds_val.output_dim},   name={ds_val.name}")
print(f"ds_test:  input_dim={ds_test.input_dim},  output_dim={ds_test.output_dim},  name={ds_test.name}")


## 3.4 Recap

After §3.1–§3.3 we have:

| Object | Type | Shape / size |
|---|---|---|
| `ds_train`, `ds_val`, `ds_test` | `NNTabularDataset` | 105 / 22 / 23 samples |
| `train_loader`, `val_loader`, `test_loader` | `torch.utils.data.DataLoader` | batches of 32 / 22 / 23 |
| `FEATURE_COLS`, `TARGET_COL`, `CLASS_NAMES` | lists | 4 / 1 / 3 |
| `scaler` | `MinMaxScaler` | fit on train only |

The features are unitless in `[0, 1]`; the target is integer-coded `{0, 1, 2}` for `{setosa, versicolor, virginica}`. §4 builds three FFN candidates over these loaders.


In [ ]:
recap = pd.DataFrame(
    {
        "split":  ["train", "val", "test"],
        "n":      [len(df_train), len(df_val), len(df_test)],
        "batch":  [ds_train.batch_sizes[0], ds_val.batch_sizes[0], ds_test.batch_sizes[0]],
        "name":   [ds_train.name, ds_val.name, ds_test.name],
    }
)
recap


# 4. Model

We compare three feed-forward topologies, all using the same `nnx.NNModel` + `Nets.FEED_FWD` + `Losses.CROSS_ENTROPY` + `Optims.ADAM` + `Devices.CPU` core. The only varying axis is `NNParams.hidden_dims`.

## 4.1 Candidate A — Linear baseline (`hidden_dims=[]`)

The simplest network the framework supports — input → linear → softmax-classifier, no hidden non-linearity. Equivalent (up to optimizer / loss-surface differences) to multinomial logistic regression. On iris this is expected to come close to the ceiling because the species are almost linearly separable from petal-length + petal-width alone (see §3.2 pairplot). Establishes the *floor* the deeper candidates have to beat.

## 4.2 Candidate B — One hidden layer (`hidden_dims=[8]`)

Adds a single 8-unit hidden layer with `LeakyReLU` (NNx default activation). 8 units = roughly 2× the input dimension — small enough that overfitting on 105 train samples is unlikely, large enough to express a moderate non-linear boundary. Tests whether *any* non-linearity helps over Candidate A.

## 4.3 Candidate C — Two hidden layers (`hidden_dims=[16, 8]`)

Two hidden layers (16 → 8). Tests whether *depth* helps over Candidate B. The funnel shape (16 → 8 → 3) is the most common iris-MLP topology in tutorial notebooks; we adopt it for ease of comparison against the source CS5644 lesson, which used `(10, 4, 5)` — a similarly modest, similarly funnel-shaped three-layer stack.

## 4.4 What we expect

The *a priori* hypothesis is that Candidate A already achieves > 95 % test accuracy and that B and C close at most a few percentage points. The §6 verdict confirms or rejects this.


In [ ]:
# Shared across all three candidates.
shared_model_params = NNModelParams(
    net=Nets.FEED_FWD,
    device=Devices.CPU,
    loss=Losses.CROSS_ENTROPY,
)

# Per-candidate NNParams (only hidden_dims varies). dropout_prob=0.0
# on the linear baseline (no hidden layer to drop). dropout_prob=0.1
# on the two MLPs — tiny model + tiny dataset means heavy dropout
# would starve them.
candidate_specs: list[dict] = [
    {
        "name": "A: linear baseline []",
        "net_params": NNParams(
            dropout_prob=0.0,
            hidden_dims=[],
            input_dim=ds_train.input_dim,
            output_dim=ds_train.output_dim,
        ),
    },
    {
        "name": "B: 1-hidden [8]",
        "net_params": NNParams(
            dropout_prob=0.1,
            hidden_dims=[8],
            input_dim=ds_train.input_dim,
            output_dim=ds_train.output_dim,
        ),
    },
    {
        "name": "C: 2-hidden [16, 8]",
        "net_params": NNParams(
            dropout_prob=0.1,
            hidden_dims=[16, 8],
            input_dim=ds_train.input_dim,
            output_dim=ds_train.output_dim,
        ),
    },
]

# Render the candidate table.
pd.DataFrame(
    [
        {
            "candidate": spec["name"],
            "hidden_dims": str(spec["net_params"].hidden_dims),
            "dropout_prob": spec["net_params"].dropout_prob,
            "n_params (rough)": sum(
                a * b for a, b in zip(spec["net_params"].dims[:-1], spec["net_params"].dims[1:])
            ),
        }
        for spec in candidate_specs
    ]
)


# 5. Training

Each candidate trains under the same `NNTrainParams` (Adam at `max_lr=1e-2`, weight decay `5e-4`, `n_epochs` resolved by `SMOKE_TEST`). We capture each candidate's `NNRun` so §6 can compare their convergence trajectories and final checkpoints.

Train + validation error curves overlay on a single `VisUtils.multi_line_plot` so visual comparison is immediate.


In [ ]:
shared_train_params = (
    NNTrainParams(
        n_epochs=n_epochs,
        optim=NNOptimParams(name=Optims.ADAM, max_lr=1e-2, weight_decay=5e-4, momentum=(0.9, 0.999)),
        seed=SEED,
    )
    .with_train_loader(value=train_loader)
    .with_val_loader(value=val_loader)
)

models: dict[str, NNModel] = {}
runs: dict[str, NNRun] = {}

for spec in candidate_specs:
    name = spec["name"]
    print(f"\n=== training candidate {name} ===")
    set_seed(SEED)  # re-pin RNGs before each candidate so identical seeds = identical init
    model = NNModel(params=shared_model_params, net_params=spec["net_params"])
    run = model.train(params=shared_train_params)
    models[name] = model
    runs[name] = run
    last_idp = run.idps[-1]
    print(
        f"  final train error = {last_idp.train_edp.error:.4f}"
        f"   final val error = {last_idp.val_edp.error:.4f}"
    )


In [ ]:
max_iters = max(runs[name].idps[-1].iter_idx for name in runs)
candidate_names = list(runs.keys())

VisUtils.multi_line_plot(
    x_ticks_inc=max(1, max_iters // 20),
    x_axis_label="Iteration",
    y_axis_label="Error",
    title="Training & validation error — candidates A / B / C",
    fig_size=(1100, 500),
    x=list(range(max_iters + 1)),
    yss_legend=[
        ["Training", "Validation"],
        candidate_names,
    ],
    yss=[
        [
            [idp.train_edp.error for idp in runs[name].idps],
            [idp.val_edp.error if idp.val_edp is not None else None for idp in runs[name].idps],
        ]
        for name in candidate_names
    ],
)
